## TIFON Databases. Read and load example databases

In [ ]:
import sys
sys.path.append('/home/m.jaraiz/repos/pyLowOrder/')
from FotR import FRODO, SAM

def read_db(datafolder, case_idx):
    db = FRODO(root_dir = datafolder, format = 'CODA', initial_parse = True)
    
    db.extract_inputs(
        id_groups = (3,),
        cases_idx = case_idx,
        vtu_type='surface',
        verbose=False
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=0, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )

    db.extract_outputs(
        id_groups=(3,),
        stage=1, cases_idx = case_idx,
        var_name_excluded = [
            'BoundaryValues_CoefSkinFrictionX',
            'BoundaryValues_CoefSkinFrictionY',
            'BoundaryValues_CoefSkinFrictionZ'
            ],
        vtu_type='surface',
        )
    
    return db

In [ ]:
# Base de datos original
case_idx = list(range(100))
fuera = [64, 79, 87, 88, 94]
for c in fuera:
    case_idx.remove(c)
db_0 = read_db(datafolder='/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3/', case_idx=case_idx)

In [ ]:
# Base de datos adicional
db_1 = read_db('/home/m.jaraiz/Documentos/DATASETS/data_TIFON/rans3_prueba/', case_idx='all')

In [ ]:
print(db_0.metadata['design_vars'])
print(db_1.metadata['design_vars'])
flcc = db_1.data_dict['CADGroup_3']['FlCc'][:, :-1]
db_1.data_dict['CADGroup_3']['FlCc'] = flcc

### Merge two sets

In [ ]:
db_completo = FRODO.merge_datasets(
    sources = [(db_0, '3'), (db_1, '3')],
    new_group_id='3_completo',
    k=4,
    mesh_ref=0,
    cache=True
    
)
db_completo.summary_data()

### Interpolation between meshes

In [ ]:
new_mesh = {
    "Coord": db_1.data_dict['CADGroup_3']['Coord'],
    "Conec": db_1.data_dict['CADGroup_3']['Conec'],
    "NodeCoord": db_1.data_dict['CADGroup_3']['NodeCoord'],
    # lo que tengas
}
db_0.sets.create_group_by_interpolation(
    id_group_src='3',
    new_group_id='3_interp',
    new_mesh=new_mesh,
    method='idw',
    k=8
    )

In [ ]:
db_0.summary_data()